# 3 · Claude Code II: The Growth Rate Hidden in Apple's Price

**Outcome of this session:** Session 2 ended with a measured premium and an open question — the market prices Apple near the top of its peer group; is that justified? Today you answer it with a **discounted cash flow (DCF) model** run *backwards*: instead of assuming a growth rate to compute a value, you solve for **the growth rate that $309 implies** — and judge whether it is plausible. Then you build the layer that lets you trust an AI-written analysis: code that machine-checks every quoted claim against its source.

**In this notebook you will:**

- Fetch Apple's real free cash flow live from its SEC filing
- Direct Claude to build a small DCF, function by function — the tests decide
- Invert it: compute the growth rate the market price assumes, and judge it
- Build the verification layer that machine-checks an AI-written analysis

## The working method: Claude writes, tests decide

For each exercise: select the function's docstring, press `Option+K` (`Alt+K` on Windows), ask *"implement this"*, read the diff, accept, and run the ✅ check below it. **Tests are the contract that lets you trust AI-written code**: each check encodes a case whose correct answer is known in advance, so acceptance depends on verification rather than on how confident the model sounds.

When something breaks, the debugging protocol:

1. **Read the traceback bottom-up**: the last line says *what* happened, the marked line says *where*.
2. **Diagnose before fixing**: make Claude explain the cause first.
3. **Change one thing at a time.**

In [ ]:
import sys, os, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
# If you pulled course updates while this kernel was running, pick them up here
# (a no-op on a fresh kernel; saves a restart otherwise):
import importlib
for _n in [n for n in list(sys.modules) if n.startswith("toolkit")]:
    importlib.reload(sys.modules[_n])
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ImportError:
    pass
HAS_KEY = bool(os.environ.get("ANTHROPIC_API_KEY"))
print(f"repo root: {ROOT}")
print(f"API key:   {'configured' if HAS_KEY else 'NOT SET - cells that call Claude will be skipped'}")

## Part A · The question Apple left open

Yesterday ended with: *the market prices Apple near the top of its peer group — is that justified?* Today's tool answers it. First, what we are doing, stated carefully, because the natural first guess is wrong:

**We are not forecasting the price from historical data.** We use exactly **one** real number — Apple's *current* yearly free cash flow, fetched from its filing below — and we reason forward from it:

1. *Imagine* that cash flow grows at some yearly rate `g` for ten years.
2. Convert each future year's cash into **today's money** (money later is worth less than money now: cash arriving in year `t` is divided by `(1 + r)ᵗ`).
3. Add it all up. Different guesses of `g` give different totals.
4. **Find the `g` that makes the total exactly equal today's price.**

That `g` is the point of the session: it is **the growth story you must believe in order to pay $309**. The market's price, translated into words. Whether the story is *plausible* is then your judgment — and last year's actual change, printed for contrast, is the first piece of evidence.

Two numbers are assumptions, stated openly and varied so you see their effect: the **required return `r` = 9%** (the yearly return an investor demands from a large, established company's stock) and a **terminal growth of 2.5%** after year ten (roughly the long-run economy — no business outgrows the economy forever).

In [ ]:
from toolkit import edgar
import pandas as pd

row = pd.read_csv(ROOT / "session-02-coding-copilot" / "data" / "tech_financials.csv").set_index("ticker").loc["AAPL"]
SHARES, PRICE = row["shares_m"] * 1e6, row["price_usd"]

try:
    print("Loading Apple's cash flows live from SEC EDGAR ...")
    facts = edgar.get_company_facts("AAPL")
    ocf = edgar.annual_values(facts, ["NetCashProvidedByUsedInOperatingActivities"], n=2)["values"]
    capex = edgar.annual_values(facts, ["PaymentsToAcquirePropertyPlantAndEquipment"], n=2)["values"]
    FCF0 = ocf[-1]["val"] - capex[-1]["val"]
    fcf_prior = ocf[0]["val"] - capex[0]["val"]
    print(f"  fiscal year ended {ocf[-1]['fy_end']}: operating cash flow {ocf[-1]['val']/1e9:,.1f}bn"
          f" - capital expenditure {capex[-1]['val']/1e9:,.1f}bn")
except Exception as exc:
    FCF0, fcf_prior = 98.8e9, 108.9e9   # bundled fallback, fiscal 2025 / 2024
    print(f"(live fetch unavailable: {type(exc).__name__} - using the bundled figures)")

print(f"\nApple's free cash flow, latest fiscal year : ${FCF0/1e9:,.1f}bn")
print(f"one year earlier                            : ${fcf_prior/1e9:,.1f}bn"
      f"   (change: {FCF0/fcf_prior-1:+.1%})")
print(f"market price ${PRICE} x {SHARES/1e9:,.1f}bn shares = market value ${SHARES*PRICE/1e12:,.2f}tn")

## Part B · Lab 1: make Claude build the model

Two small functions make the whole DCF. For each: select the docstring, `Option+K`, *"implement this"*, read the diff, run the check. The checks use toy numbers whose correct answers are known exactly, so you never take the code on faith.

### Exercise 1: project the cash flows

In [ ]:
def project_fcf(fcf0: float, growth: float, years: int) -> list[float]:
    """Imagine the cash flow growing: each year is the previous year times (1 + growth)."""
    flows = []
    fcf = fcf0
    for year in range(years):
### START CODE HERE ###
        fcf = fcf * (1 + None)      # each year grows the previous one by what?
### END CODE HERE ###
        flows.append(fcf)
    return flows

In [ ]:
# ✅ self-check: run me (offline). 100 growing at 10% for 3 years is 110, 121, 133.1.
flows = project_fcf(100.0, 0.10, 3)
assert len(flows) == 3, "one entry per year"
assert abs(flows[0] - 110.0) < 1e-9 and abs(flows[2] - 133.1) < 1e-9, "year t is fcf0 x (1+g)^t"
print("All checks passed ✅")

### Exercise 2: discount to today

Money later is worth less than money now: a flow in year *t* is divided by `(1 + r) ** t`. After the last projected year, the **terminal value** captures everything beyond: `final flow × (1 + terminal growth) ÷ (r − terminal growth)` — and it sits at the end of year N, so it is discounted by `(1 + r) ** N` too.

The check below is exact by design: two flat flows of 100 at r = 10% with zero terminal growth are worth precisely **1000.00** — computable by hand, so if the code disagrees, the code is wrong.

In [ ]:
def present_value(flows: list[float], r: float, terminal_growth: float) -> float:
    """Convert future cash into today's money, year by year, plus the terminal value."""
    value_today = 0.0
    year = 0
    for flow in flows:
        year = year + 1
### START CODE HERE ###
        value_today = value_today + flow / (1 + r) ** None    # divide by (1+r), compounded how many times?
### END CODE HERE ###

    # After the last year: the terminal value (a perpetuity), also brought to today's money.
### START CODE HERE ###
    terminal = flows[-1] * (1 + terminal_growth) / (None - None)   # the perpetuity denominator
### END CODE HERE ###
    value_today = value_today + terminal / (1 + r) ** len(flows)
    return value_today

In [ ]:
# ✅ self-check: run me (offline, exact). Two flows of 100, r=10%, terminal growth 0:
# by hand: 100/1.1 + 100/1.21 = 173.55…, terminal 100/0.10 = 1000 discounted /1.21 = 826.44…
# together exactly 1000.00.
value = present_value([100.0, 100.0], 0.10, 0.0)
assert abs(value - 1000.0) < 0.01, f"got {value:.2f}, expected exactly 1000.00 - check year-1 discounting and the terminal"
print("All checks passed ✅  Both functions verified. Now the question:")

In [ ]:
def implied_growth(target_value: float, fcf0: float, r: float,
                   terminal_growth: float = 0.025, years: int = 10) -> float:
    """Solve (by bisection) for the growth rate that makes the DCF equal the target. GIVEN."""
    lo, hi = -0.5, 0.5
    for _ in range(80):
        mid = (lo + hi) / 2
        value = present_value(project_fcf(fcf0, mid, years), r, terminal_growth)
        lo, hi = (mid, hi) if value < target_value else (lo, mid)
    return (lo + hi) / 2

market_value = SHARES * PRICE
print("What growth does the market price assume? (10-year projection, 2.5% terminal)\n")
for r in (0.08, 0.09, 0.10):
    g = implied_growth(market_value, FCF0, r)
    print(f"  required return {r:.0%}  ->  implied free-cash-flow growth {g:6.1%} per year, for ten years")
g9 = implied_growth(market_value, FCF0, 0.09)
print(f"\nAt the central assumption (9%): the price of ${PRICE} says Apple's free cash flow")
print(f"grows {g9:.1%} every year for a decade. Last year it actually changed by {FCF0/fcf_prior-1:+.1%}.")

*Reading the result, plainly.* Nothing was predicted here: the model **translated the price into the assumption it contains**. The market's $309 is a belief about the future, made visible: at a 9% required return, it says Apple's free cash flow will grow about **17% per year, every year, for ten years**. Last year it actually *fell* about 9%.

Three things this teaches:

1. **Session 2's premium now has a precise meaning.** "Priced near best-in-group" translates to a concrete claim: a decade of mid-teens cash-flow growth. The comparable analysis measured the premium; the reverse DCF states what believing it requires.
2. **The answer moves with the assumption, and honest work shows that.** The table prints the implied growth at 8%, 9% and 10% required returns — a range, with its driver visible, exactly as in Session 2.
3. **The model does not say buy or sell.** Whether a company that just shrank its free cash flow can grow it 17% a year for ten years is a judgment about products, competition and pricing power — the analyst's question. The model's contribution is that the question is now *specific*.

You built both functions with Claude, and accepted them because the checks passed — including one whose answer you could verify by hand. That is the working relationship this course trains: **the model writes, the tests decide, you judge.**

## Part C · Lab 2: can you trust an AI-written analysis?

The reverse DCF left you with a specific question: *can Apple really grow free cash flow ~17% a year for a decade?* Answering it needs **evidence**, and the richest regular source of evidence is the **earnings call** — the quarterly conference where management presents results and analysts ask questions. An AI can read a whole call and draft the analysis in seconds. Which raises this session's second trust problem: **how do you know the AI did not invent the evidence?**

This engine is exactly the tool you would point at Apple's next earnings call to gather evidence on the growth question. We *train* it on a **fictional company's call**, for one reason stated openly: a real company's calls are in the model's training data, so only a fictional one proves that every claim comes from the *document* rather than from memory. The engine runs unchanged on any real transcript.

In [ ]:
sys.path.insert(0, str(ROOT / "session-03-debugging" / "lab"))
import earnings_starter as engine
from earnings_starter import EARNINGS_SCHEMA, analyze, DEFAULT_TRANSCRIPT
from toolkit import llm            # for llm.show(): renders long output readably

# The grounding rules, applied at the source (the same discipline as Session 1):
engine.SYSTEM = """You are a buy-side equity analyst preparing an internal note.
- Use ONLY the transcript provided. No outside knowledge, no memory of other companies.
- Every evidence_quote must be VERBATIM from the transcript; each will be machine-checked.
- If the transcript does not support a claim, do not make it.
- Separate management's framing from fact; note what guidance excludes."""

print(f"Loading the earnings-call transcript from a file bundled with the course:")
print(f"  {DEFAULT_TRANSCRIPT.relative_to(ROOT)}")
transcript = DEFAULT_TRANSCRIPT.read_text()
print(f"  loaded: {len(transcript.split()):,} words | speakers: CEO, CFO, five analysts | 7 planted red flags")
print()
print("How you would obtain a real one: earnings calls are NOT on EDGAR. Companies")
print("post recordings and prepared remarks on their investor-relations pages, and")
print("transcript providers offer them through paid APIs. The engine below runs")
print("unchanged on any transcript you drop into a text file.")

### Exercise 3: verify_evidence, the fabrication detector

Every claim the model makes carries a verbatim quote. You check each quote against the source **in plain Python**. Normalize both sides first (collapse whitespace, lowercase, straighten curly quotes) so that formatting differences cannot cause false negatives. Set `item["verified"]` on every item in `key_themes`, `risks`, `red_flags`, and store totals in `analysis["_verification"]`.

In [ ]:
import re

def _normalize(text: str) -> str:
    """Whitespace-collapse + casefold + straighten curly quotes. GIVEN - it's
    plumbing; YOUR work is the verification logic below."""
    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')
    return re.sub(r"\s+", " ", text).casefold().strip()

def verify_evidence(analysis: dict, transcript: str) -> dict:
### START CODE HERE ###
    haystack = _normalize(None)                        # normalize which text?
    checked = failed = 0
    for section in ("key_themes", "risks", "red_flags"):
        for item in analysis.get(section, []):
            quote = item.get("evidence_quote", "")
            item["verified"] = bool(quote) and None in haystack   # hint: the NORMALIZED quote
            checked += 1
            failed += 0 if item["verified"] else 1
    analysis["_verification"] = {"quotes_checked": None, "quotes_failed": None}
### END CODE HERE ###
    return analysis

print("defined - now catch a fabrication:")

In [ ]:
# ✅ self-check: run me. The canned dry-run analysis hides ONE deliberately
# fabricated quote. If your verify_evidence works, it catches exactly that one.
analysis = verify_evidence(analyze(transcript, dry_run=True), transcript)
v = analysis["_verification"]
print(f"quotes checked: {v['quotes_checked']}, failed: {v['quotes_failed']}")
assert v["quotes_checked"] >= 10, "check key_themes, risks AND red_flags"
assert v["quotes_failed"] == 1, "exactly ONE quote is fabricated - if 0, your matching is too loose; if >1, normalize better"
fake = [t for s in ("key_themes", "risks", "red_flags") for t in analysis[s] if not t["verified"]]
print("All checks passed ✅  Caught fabrication:", repr(fake[0]["evidence_quote"]))

**The same contract, in the industry's words.** The engine's schema is a raw JSON dictionary (JSON is the standard text format for structured data); in professional Python the standard way to declare and enforce such a contract is **Pydantic**: one class per object, one typed field per key. The cell below declares the analysis schema as Pydantic models and re-validates the model output through it — the same gate, now typed. Everything downstream could then use `analysis.red_flags[0].flag` instead of dictionary keys, and any malformed reply raises a precise `ValidationError` naming the offending field.

In [ ]:
from pydantic import BaseModel

class Theme(BaseModel):
    theme: str
    evidence_quote: str
    verified: bool | None = None

class Risk(BaseModel):
    risk: str
    severity: str
    evidence_quote: str
    verified: bool | None = None

class RedFlag(BaseModel):
    flag: str
    why_it_matters: str
    evidence_quote: str
    verified: bool | None = None

class EarningsAnalysis(BaseModel):
    overall_sentiment: str
    key_themes: list[Theme]
    risks: list[Risk]
    red_flags: list[RedFlag]

typed = EarningsAnalysis.model_validate(analysis)   # raises ValidationError on any shape violation
print(f"Validated: {len(typed.key_themes)} themes, {len(typed.risks)} risks, "
      f"{len(typed.red_flags)} red flags - all typed.")
print("First red flag, by attribute access:", typed.red_flags[0].flag)

That quote, a promised margin recovery, is entirely plausible and appears nowhere in the transcript. **Reading alone would likely have missed it; your code did not.** This is verification in practice.

**What to expect on a live run.** The cell below also runs the engine against the live Claude API (application programming interface) if you have a key. There the model writes its own quotes, and a current model quotes accurately most of the time, so expect **all or nearly all** quotes to verify. If one or two are flagged, that is the checker doing its job: the model paraphrased instead of quoting verbatim — read the flagged lines and judge them yourself. The planted fabrication exists in the canned analysis precisely so that every student sees a catch at least once. In production the value of this layer is not that it fires often; it is that nothing reaches a committee unchecked.

In [ ]:
def memo_from(analysis: dict, source_name: str) -> str:
    """A compact analyst note: every claim printed with its verification mark."""
    lines = [f"# Meridian Semiconductor - Q2 FY2026 note (draft, from {source_name})",
             f"\n**Sentiment:** {analysis['overall_sentiment']} - {analysis['sentiment_rationale']}\n"]
    for section, key in [("Key themes", "theme"), ("Risks", "risk"), ("Red flags", "flag")]:
        lines.append(f"## {section}\n")
        for it in analysis[section.lower().replace(" ", "_")]:
            mark = "verified" if it.get("verified") else "NOT FOUND IN TRANSCRIPT"
            lines.append(f"- **{it[key]}** [{mark}]")
            lines.append(f"  - evidence: \"{it['evidence_quote'][:140]}\"")
    v = analysis["_verification"]
    lines.append(f"\n*Evidence check: {v['quotes_checked'] - v['quotes_failed']}/{v['quotes_checked']} quotes verified against the source.*")
    return "\n".join(lines)

OUTD = ROOT / "outputs"; OUTD.mkdir(exist_ok=True)
(OUTD / "meridian_earnings_memo.md").write_text(memo_from(analysis, "dry-run"))
print("outputs/meridian_earnings_memo.md written (dry-run).\n")

if HAS_KEY:
    live = verify_evidence(analyze(transcript, dry_run=False), transcript)
    lv = live["_verification"]
    (OUTD / "meridian_earnings_memo.md").write_text(memo_from(live, "LIVE"))
    print(f"LIVE run: {lv['quotes_checked'] - lv['quotes_failed']}/{lv['quotes_checked']} quotes verified.")
    print("0 failures is the expected result with a current model: the checker reports, it does not accuse.\n")
    llm.show(memo_from(live, "LIVE"), title="Your memo, every claim marked")
else:
    print("No API key - the dry-run memo still demonstrates the whole pipeline.")

## Deliverable checklist

- [ ] Both DCF functions written by Claude and accepted only after their checks passed
- [ ] You can state the session's finding in one sentence: what growth the market price implies, and what free cash flow actually did last year
- [ ] Your `verify_evidence` catches **exactly 1** fabricated quote in dry-run
- [ ] `outputs/meridian_earnings_memo.md` generated (live if you have a key) and committed to your repo

**Next:** `04-workflows-edgar.ipynb`, where the by-hand steps of these two days become one automated workflow over live filings.